## Installing dependencies
Here, the libraries needed for the project are installed:
- Transformers → gives access to GPT‑2 and other models
- Datasets → helps load and process text data
- Torch → provides deep learning support


In [5]:
!pip install transformers datasets torch


## Preparing dataset
Here a small text file with sample sentences are created.
This acts as my training data for fine‑tuning GPT‑2.
Later, it can be replaced with a larger dataset.


In [6]:
import os
os.makedirs("data", exist_ok=True)

sample_text = """
The future of AI is collaborative and human-centered.
Consistency beats motivation—show up, do small steps, and improve.
Skincare routines work best when kept simple and sustainable.
"""
with open("data/train.txt", "w") as f:
    f.write(sample_text.strip())


## **Loading dataset and tokenizer**

Here the dataset is loaded into HuggingFace Datasets. The GPT‑2 tokenizer is also initialized to convert text into tokens.

In [7]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

dataset = load_dataset("text", data_files={"train": "data/train.txt"})

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

## **Tokenizing dataset**

Here the text is tokenized and labels are created. Labels are copies of the input tokens so GPT‑2 can calculate loss during training.

In [8]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

## **Loading model and training arguments**

Here the GPT‑2 model is loaded. Training parameters such as batch size, number of epochs, and logging options are set.

In [9]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments

# Load GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.eos_token_id

# Training configuration
training_args = TrainingArguments(
    output_dir="results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    save_steps=500,
    save_total_limit=2,
    logging_dir="logs",
    logging_steps=50,
    fp16=True
)


## **Training the model**

Here the HuggingFace Trainer is used to fine‑tune GPT‑2 on the dataset. The model learns patterns from the text.


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"]
)

trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=3, training_loss=5.99687385559082, metrics={'train_runtime': 75.4872, 'train_samples_per_second': 0.119, 'train_steps_per_second': 0.04, 'total_flos': 587907072000.0, 'train_loss': 5.99687385559082, 'epoch': 3.0})

## **Generating text**

Here the fine‑tuned model is tested with a prompt. The model generates new text based on what it learned.

In [11]:
prompt = "The future of AI is"
inputs = tokenizer.encode(prompt, return_tensors="pt")
outputs = model.generate(
    inputs,
    max_length=60,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The future of AI is a lot more complicated.

In the future, AI could be more important than we realize.


## **Saving results**

Here the generated text is saved into a results folder. This makes it easy to upload outputs to GitHub and share them.

In [12]:
import os
os.makedirs("results", exist_ok=True)
with open("results/samples.txt", "w") as f:
    f.write(tokenizer.decode(outputs[0], skip_special_tokens=True))
